# Render and Extract UF3 Snapshots

Notebook version of `render_and_extract.py` -- renders every generated story
listed in the UF3 manifest (written by `storybook-generate-stories.ipynb`)
via a running Storybook instance and extracts a DOM layout+style snapshot per
story using Playwright.

**Prerequisite:** Storybook must already be running and fully indexed
(e.g. `npm run storybook` inside `dataset/storybook`), reachable at
`STORYBOOK_URL`. This notebook only consumes an already-running dev server --
it does not start one.

Since multiple generated stories can share one `importPath` (several named
exports injected into the same existing GT story file), the actual live
story ID is resolved at runtime by fetching Storybook's own `/index.json`
and matching `(importPath, normalized exportName)` against it -- never
predicted, so it is robust against Storybook's internal ID-sanitization
behaviour.

In [7]:
import re
import sys
import json
import time
import asyncio
import threading
import urllib.request
import urllib.error
from pathlib import Path

from playwright.async_api import (
    async_playwright,
    Error as PlaywrightError,
    TimeoutError as PlaywrightTimeoutError,
)

## 0. Repo root and configuration

In [8]:
def find_repo_root(marker: str = 'dataset', start: Path | None = None) -> Path:
    """Searches upward from the current working directory until a folder
    named 'marker' is found -- robust against the kernel's working directory
    not matching the notebook's own folder."""
    start = start or Path.cwd()

    for parent in [start, *start.parents]:
        if (parent / marker).is_dir():
            return parent

    raise FileNotFoundError(f"Could not find a folder named '{marker}' above {start}")


REPO_ROOT = find_repo_root()

# 'components' or 'uis'
TYPE = 'components'

STORYBOOK_URL = 'http://localhost:6006'
WAIT_MS = 200               # extra settle time after networkidle, for CSS transitions/animations
VIEWPORT_WIDTH = 1280
VIEWPORT_HEIGHT = 800

MANIFEST_PATH = REPO_ROOT / 'evaluations' / f'stories_manifest_{TYPE}.json'
SNAPSHOTS_DIR = REPO_ROOT / 'evaluations' / 'visual_fidelity_snapshots' / TYPE

print(f'MANIFEST_PATH: {MANIFEST_PATH}  (exists: {MANIFEST_PATH.exists()})')
print(f'SNAPSHOTS_DIR: {SNAPSHOTS_DIR}')

MANIFEST_PATH: C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\stories_manifest_components.json  (exists: True)
SNAPSHOTS_DIR: C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\visual_fidelity_snapshots\components


## 1. DOM extraction script

Embedded directly (identical to `extraction.js`) rather than loaded from a
sibling file, so the notebook is self-contained.

In [9]:
EXTRACTION_JS = """
() => {
  const root = document.querySelector('#storybook-root') || document.body;
  const STYLE_PROPS = ['color', 'backgroundColor', 'fontSize', 'fontWeight', 'padding', 'margin',
                        'borderRadius', 'display', 'justifyContent', 'alignItems', 'gap',
                        'borderColor', 'borderWidth', 'textAlign'];
  const elements = [];

  function walk(node, pathArr, depth) {
    if (node.nodeType !== 1) return;

    const rect = node.getBoundingClientRect();
    const cs = getComputedStyle(node);
    const styles = {};
    for (const p of STYLE_PROPS) styles[p] = cs[p];

    const ownText = Array.from(node.childNodes)
      .filter((n) => n.nodeType === 3)
      .map((n) => n.textContent.trim())
      .filter(Boolean)
      .join(' ')
      .trim();

    elements.push({
      tag: node.tagName,
      pc_name: node.getAttribute('data-pc-name'),
      pc_section: node.getAttribute('data-pc-section'),
      text: ownText,
      bbox: { x: rect.x, y: rect.y, width: rect.width, height: rect.height },
      styles,
      depth,
      path: pathArr.join('.'),
    });

    Array.from(node.children).forEach((child, i) => walk(child, [...pathArr, i], depth + 1));
  }

  walk(root, [0], 0);

  return { elements, viewport: { width: window.innerWidth, height: window.innerHeight } };
}
"""

## 2. Storybook index resolution

In [10]:
def check_storybook_reachable(base_url: str, timeout: float = 5.0) -> bool:
    try:
        with urllib.request.urlopen(f'{base_url}/index.json', timeout=timeout):
            return True
    except (urllib.error.URLError, TimeoutError):
        return False


def fetch_live_index(base_url: str, timeout: float = 10.0) -> dict:
    with urllib.request.urlopen(f'{base_url}/index.json', timeout=timeout) as resp:
        raw = json.loads(resp.read().decode('utf-8'))

    return raw.get('entries', raw.get('stories', {}))


def _normalize_for_match(s: str) -> str:
    """Lowercases and strips everything but [a-z0-9] -- used to compare our
    own export names (already pure [a-z0-9_]) against Storybook's
    human-readable story 'name' field, which may re-case/space/underscore
    the identifier differently. Both sides reduce to the same character
    sequence regardless of that formatting, without needing to replicate
    Storybook's exact transform."""
    return re.sub(r'[^a-z0-9]', '', s.lower())


def resolve_story_ids(manifest: list[dict], index: dict) -> tuple[list[dict], list[dict]]:
    """Annotates each manifest entry with a resolved live 'storyId'.

    Returns (resolved, unresolved) -- unresolved entries are reported and
    skipped rather than guessed at.
    """
    by_import_path: dict[str, list[dict]] = {}
    for entry_id, entry in index.items():
        by_import_path.setdefault(entry.get('importPath', ''), []).append({**entry, 'id': entry.get('id', entry_id)})

    resolved, unresolved = [], []

    for m in manifest:
        candidates = by_import_path.get(m['importPath'], [])
        target = _normalize_for_match(m['exportName'])

        match = next((c for c in candidates if _normalize_for_match(c.get('name', '')) == target), None)

        if match is None:
            unresolved.append(m)
            continue

        resolved.append({**m, 'storyId': match['id']})

    return resolved, unresolved

## 3. Snapshot extraction and output path

In [11]:
async def extract_snapshot(page, story_id: str, base_url: str, wait_ms: int = 200,
                            viewport: dict | None = None, max_retries: int = 2) -> dict | None:
    """viewport (optional): {'width': int, 'height': int} -- if given, set on
    the page BEFORE navigation, so the mockup's own intended size (see
    storybook-generate-stories.ipynb, '1b. Viewport per mockup') is used
    instead of whatever the page's default viewport happens to be. GT and
    every generated variant of the same mockup receive the identical value,
    read from the same manifest entry field.

    Retries the full goto+evaluate sequence up to max_retries times on any
    Playwright error (not just timeouts) -- both a destroyed execution
    context during evaluate() (typically Storybook's HMR websocket triggering
    a reload mid-read) and a navigation error like net::ERR_ABORTED during
    goto() are usually transient at this scale (hundreds to thousands of
    stories in one run) rather than a permanent problem with that specific
    story. Critically, catching only PlaywrightTimeoutError here (as before)
    left net::ERR_ABORTED and similar navigation errors completely unhandled --
    an uncaught PlaywrightError would propagate out of this function and
    crash the entire run_extraction() loop, not just skip the one story.
    """
    if viewport is not None:
        await page.set_viewport_size(viewport)

    url = f'{base_url}/iframe.html?id={story_id}&viewMode=story'

    for attempt in range(1, max_retries + 1):
        is_last_attempt = attempt == max_retries

        try:
            await page.goto(url, wait_until='networkidle', timeout=60000)
        except PlaywrightTimeoutError:
            print(f'  TIMEOUT loading {story_id} (attempt {attempt}/{max_retries})')
            if is_last_attempt:
                return None
            await page.wait_for_timeout(500)
            continue
        except PlaywrightError as e:
            print(f'  NAV ERROR loading {story_id} (attempt {attempt}/{max_retries}): {e}')
            if is_last_attempt:
                return None
            await page.wait_for_timeout(500)
            continue

        await page.wait_for_timeout(wait_ms)

        try:
            data = await page.evaluate(EXTRACTION_JS)
        except PlaywrightError as e:
            print(f'  ERROR extracting {story_id} (attempt {attempt}/{max_retries}): {e}')
            if is_last_attempt:
                return None
            await page.wait_for_timeout(500)
            continue

        if not data.get('elements'):
            print(f'  WARNING: {story_id} produced zero elements -- story may not have rendered '
                  f'(check for a Storybook error boundary / console error).')

        return data

    return None


def snapshot_output_path(snapshots_dir: Path, entry: dict) -> Path:
    """'components':       .../{approach}/{prompt_strategy}/{complexity}/{stem}.json
    'uis':                .../{approach}/{prompt_strategy}/{variant}/{stem}.json
    'a'  (components):     .../a/{complexity}/{stem}.json         (no prompt_strategy)
    'a'  (uis):            .../a/{variant}/{stem}.json            (no prompt_strategy)
    'gt' (components):     .../gt/{complexity}/{stem}.json        (no prompt_strategy)
    'gt' (uis):            .../gt/{stem}.json                     (no prompt_strategy,
                           no variant subfolder either -- GT is variant-
                           independent, just like the Figma JSON/screenshot
                           it comes from).

    Approach 'a' is rule-based/deterministic and, like GT, genuinely has no
    prompt_strategy (real None, not an invented placeholder -- see
    storybook-generate-stories.ipynb). Both are handled by checking the
    actual prompt_strategy value rather than enumerating approach names by
    hand, so this stays correct if another no-prompt-strategy approach is
    ever added.
    """
    group = entry.get('complexity') or entry.get('variant')

    if entry['prompt_strategy'] is None:
        if group:
            return snapshots_dir / entry['approach'] / group / f'{entry["stem"]}.json'
        return snapshots_dir / entry['approach'] / f'{entry["stem"]}.json'

    return (snapshots_dir / entry['approach'] / entry['prompt_strategy'] /
            (group or 'unknown') / f'{entry["stem"]}.json')

## 4. Main loop

In [12]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'{MANIFEST_PATH} not found. Run storybook-generate-stories.ipynb first.')

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
print(f'Loaded manifest: {len(manifest)} generated stories from {MANIFEST_PATH}')

if not check_storybook_reachable(STORYBOOK_URL):
    raise RuntimeError(
        f'Storybook not reachable at {STORYBOOK_URL}. '
        f'Start it first (e.g. `npm run storybook` in dataset/storybook).'
    )

index = fetch_live_index(STORYBOOK_URL)
manifest, unresolved = resolve_story_ids(manifest, index)

if unresolved:
    print(f'WARNING: {len(unresolved)} / {len(unresolved) + len(manifest)} manifest entries could not be '
          f'resolved against the live Storybook index and will be skipped:')
    for u in unresolved[:10]:
        print(f"    {u['storyFile']}  (exportName={u['exportName']!r})")
    if len(unresolved) > 10:
        print(f'    ... and {len(unresolved) - 10} more')
    print('  This usually means storybook-generate-stories.ipynb was run after Storybook last '
          'indexed, or against a different Storybook build than the one currently running -- '
          'restart Storybook (or wait for it to re-index) and re-run this cell.')
else:
    print(f'OK: all {len(manifest)} manifest entries resolved against the live Storybook index.')

SNAPSHOTS_DIR.mkdir(parents=True, exist_ok=True)


async def run_extraction():
    ok_count = fail_count = 0
    start_time = time.time()

    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page(viewport={'width': VIEWPORT_WIDTH, 'height': VIEWPORT_HEIGHT})

        for i, entry in enumerate(manifest, 1):
            entry_viewport = None
            if 'viewportWidth' in entry and 'viewportHeight' in entry:
                entry_viewport = {'width': entry['viewportWidth'], 'height': entry['viewportHeight']}

            snapshot = await extract_snapshot(page, entry['storyId'], STORYBOOK_URL, WAIT_MS, entry_viewport)

            if snapshot is None:
                fail_count += 1
                continue

            out_path = snapshot_output_path(SNAPSHOTS_DIR, entry)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            out_path.write_text(json.dumps({**snapshot, 'meta': entry}, indent=2), encoding='utf-8')

            ok_count += 1
            if i % 25 == 0 or i == len(manifest):
                print(f'  [{i}/{len(manifest)}] {ok_count} ok, {fail_count} failed')

        await browser.close()

    elapsed = time.time() - start_time
    print(f'\nDone in {elapsed:.1f}s: {ok_count} snapshots saved to {SNAPSHOTS_DIR}, '
          f'{fail_count} failed, {len(unresolved)} unresolved.')


def run_async_in_thread(coro_func):
    """Runs an async function to completion in a dedicated thread with its
    own freshly created event loop.

    Works around two Jupyter/Windows-specific issues at once:
    1. Playwright's sync API cannot run inside the kernel's own already-
       running asyncio loop (Jupyter kernels always have one) -- hence async
       Playwright in the first place.
    2. On Windows, Playwright's async driver needs subprocess support, which
       requires the ProactorEventLoop -- but ipykernel's own loop (already
       running by the time notebook code executes) is typically a
       SelectorEventLoop, which raises NotImplementedError for subprocess
       creation on Windows, and its policy cannot be changed after the fact.

    Running in a separate thread sidesteps both: the thread's loop is neither
    already running (no sync-API conflict) nor bound to the kernel's policy
    (so ProactorEventLoop can be set here on Windows), all without touching
    the kernel's own event loop at all.
    """
    result: dict = {}
    error: dict = {}

    def _target():
        if sys.platform == 'win32':
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        try:
            result['value'] = loop.run_until_complete(coro_func())
        except Exception as e:
            error['value'] = e
        finally:
            loop.close()

    thread = threading.Thread(target=_target)
    thread.start()
    thread.join()

    if 'value' in error:
        raise error['value']

    return result.get('value')


run_async_in_thread(run_extraction)

#cd C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo
# Powershell: .venv\Scripts\Activate.ps1 ODER CMD: .venv\Scripts\activate.bat
# playwright install chromium

Loaded manifest: 1680 generated stories from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\stories_manifest_components.json
OK: all 1680 manifest entries resolved against the live Storybook index.
  [25/1680] 25 ok, 0 failed
  ERROR extracting components-hard-1-edit-dialog--c-zero-shot-1-c-2-gpt-5-6-terra-1 (attempt 1/2): Page.evaluate: Execution context was destroyed, most likely because of a navigation
  [50/1680] 50 ok, 0 failed
  NAV ERROR loading components-hard-10-admin-panel-list--b-few-shot-10-b-2-gpt-5-6-terra-1 (attempt 1/2): Page.goto: net::ERR_ABORTED at http://localhost:6006/iframe.html?id=components-hard-10-admin-panel-list--b-few-shot-10-b-2-gpt-5-6-terra-1&viewMode=story
Call log:
  - navigating to "http://localhost:6006/iframe.html?id=components-hard-10-admin-panel-list--b-few-shot-10-b-2-gpt-5-6-terra-1&viewMode=story", waiting until "networkidle"

  [75/1680] 75 ok, 0 failed
  [100/1680] 100 ok, 0 failed
  [125/1680] 125 ok, 0 failed
  ERROR extract